**ChatGPT-4.1-mini(API)를 활용한 뉴스 생성 요약**

In [ ]:
#구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 패키지 설치 (Jupyter/Colab에서 실행)
# ═══════════════════════════════════════════════════════════════════════
!pip install pandas
!pip install openai
!pip install tqdm
!pip install nest-asyncio

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 뉴스 3줄 요약 분석기 (News Summary Generator)
# ═══════════════════════════════════════════════════════════════════════
import pandas as pd
from openai import AsyncOpenAI
import asyncio
import json
import os
import logging
import time
from typing import Tuple, Dict
import re
from tqdm import tqdm

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 1. 프롬프트 템플릿
# ═══════════════════════════════════════════════════════════════════════
PROMPT_TEMPLATE = """다음 뉴스 기사를 3문장으로 요약하고, 하나의 문자열로 구성하라.

[뉴스 기사]:
{text}

**요약 작성 규칙:**
1. 5W1H(누가, 무엇을, 언제, 어디서, 왜, 어떻게)를 파악하라.
2. 파악한 요소를 논리적 서사로 결합하라.
3. 중립적 톤을 유지하라.
4. 문어체로 응답하라.
5. 정확히 3문장으로 작성하라.

**반드시 다음 JSON 형식으로만 답변하라:**
{{
  "summary": "첫 번째 문장. 두 번째 문장. 세 번째 문장."
}}

답변 주의사항:
- summary 필드에 3문장을 하나의 문자열로 작성.
- 각 문장은 마침표(.)로 구분.
- 경어 사용 금지.
- JSON 외 다른 내용 금지.

JSON 응답:"""

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2. JSON 파서
# ═══════════════════════════════════════════════════════════════════════
class EnhancedJSONParser:
    """강화된 JSON 파싱 클래스"""

    @staticmethod
    def extract_json_robust(response_text: str) -> Dict:
        """강화된 JSON 추출 로직"""
        if not response_text or not response_text.strip():
            return EnhancedJSONParser._get_default_result("빈 응답")

        response_text = response_text.strip()

        # 1단계: 직접 JSON 파싱 시도
        try:
            if response_text.startswith('{') and response_text.endswith('}'):
                return json.loads(response_text)
        except json.JSONDecodeError:
            pass

        # 2단계: 마크다운 JSON 블록 추출
        json_patterns = [
            r'```json\s*\n?(.*?)\n?```',
            r'```\s*\n?(.*?)\n?```',
            r'`{3,}\s*json\s*\n?(.*?)\n?`{3,}',
        ]

        for pattern in json_patterns:
            match = re.search(pattern, response_text, re.DOTALL | re.IGNORECASE)
            if match:
                json_str = match.group(1).strip()
                try:
                    return json.loads(json_str)
                except json.JSONDecodeError:
                    continue

        # 3단계: 중괄호 블록 추출
        brace_patterns = [
            r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}',
            r'\{(?:[^{}]|{[^{}]*})*\}',
        ]

        for pattern in brace_patterns:
            matches = re.findall(pattern, response_text)
            for match in matches:
                try:
                    result = json.loads(match)
                    if 'summary' in result:
                        return result
                except json.JSONDecodeError:
                    continue

        # 4단계: summary 필드 직접 추출
        summary_patterns = [
            r'"summary"\s*:\s*"([^"]*(?:\\.[^"]*)*)"',
            r'summary["\']?\s*[:=]\s*["\']([^"\']+)["\']',
        ]

        for pattern in summary_patterns:
            match = re.search(pattern, response_text, re.IGNORECASE | re.DOTALL)
            if match:
                return {'summary': match.group(1).strip()}

        # 5단계: 기본값 반환
        return EnhancedJSONParser._get_default_result("파싱 실패")

    @staticmethod
    def _get_default_result(reason: str) -> Dict:
        """기본 결과 반환"""
        return {
            'summary': f'요약 실패: {reason}'
        }

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 3. 뉴스 요약 분석기
# ═══════════════════════════════════════════════════════════════════════
class NewsSummaryAnalyzer:
    """뉴스 3줄 요약 분석기"""

    def __init__(self, api_key, model, output_dir='.',
                 concurrent_requests=3, rate_limit_delay=1.0, batch_size=5):
        self.client = AsyncOpenAI(api_key=api_key)
        self.model = model
        self.output_dir = output_dir
        self.concurrent_requests = concurrent_requests
        self.rate_limit_delay = rate_limit_delay
        self.batch_size = batch_size
        self.total_cost = 0.0
        self.semaphore = asyncio.Semaphore(concurrent_requests)
        self.json_parser = EnhancedJSONParser()

        # 통계
        self.parsing_successes = 0
        self.parsing_failures = 0

        # 로깅 설정 (WARNING 레벨로 최소화)
        logging.basicConfig(level=logging.WARNING, format='%(message)s')
        self.logger = logging.getLogger(__name__)

        os.makedirs(self.output_dir, exist_ok=True)

    def combine_title_text(self, title: str, text: str) -> str:
        """제목과 본문을 결합"""
        if pd.isna(title):
            title = ""
        if pd.isna(text):
            text = ""

        title = str(title).strip()
        text = str(text).strip()

        combined = f"{title}\n\n{text}" if title and text else (title or text)
        return combined[:4000]  # 토큰 제한을 위한 길이 제한

    async def analyze_single_article(self, title: str, text: str) -> Tuple[str, float]:
        """단일 기사 분석 - 각 호출은 완전히 독립적"""

        async with self.semaphore:
            try:
                # 제목과 본문 결합
                combined_text = self.combine_title_text(title, text)

                if not combined_text or combined_text == "":
                    return "요약 실패: 빈 텍스트", 0.0

                # 프롬프트 생성 (매번 독립적으로 생성!중요!)
                prompt = PROMPT_TEMPLATE.format(text=combined_text)

                # API 호출 (각 호출은 독립적인 세션)
                response = await self.client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "user", "content": prompt}
                    ],
                    max_tokens=300,
                    temperature=0,
                    response_format={"type": "json_object"}
                )

                response_text = response.choices[0].message.content.strip()
                result = self.json_parser.extract_json_robust(response_text)

                # 결과 추출
                summary = result.get("summary", "요약 실패")

                # 파싱 성공/실패 통계
                if "요약 실패" in summary:
                    self.parsing_failures += 1
                else:
                    self.parsing_successes += 1

                # 비용 계산
                estimated_cost = 0.002
                self.total_cost += estimated_cost

                return summary, estimated_cost

            except Exception as e:
                self.parsing_failures += 1
                return f"오류: {str(e)[:50]}", 0.0

            finally:
                await asyncio.sleep(self.rate_limit_delay)

    async def process_batch(self, batch_df) -> Tuple[list, list]:
        """배치 처리"""
        tasks = []

        for _, row in batch_df.iterrows():
            task = self.analyze_single_article(
                row['title'],
                row['text']
            )
            tasks.append(task)

        results = await asyncio.gather(*tasks, return_exceptions=True)

        summaries = []
        costs = []

        for result in results:
            if isinstance(result, Exception):
                summaries.append(f"배치 오류: {str(result)[:50]}")
                costs.append(0.0)
                self.parsing_failures += 1
            else:
                summaries.append(result[0])
                costs.append(result[1])

        return summaries, costs

    async def analyze_dataframe(self, df, output_filename="news_summary_results.csv"):
        """전체 데이터프레임 분석"""

        output_path = os.path.join(self.output_dir, output_filename)

        all_summaries = []
        all_costs = []

        start_time = time.time()

        # tqdm 진행바 생성
        pbar = tqdm(total=len(df), desc="뉴스 요약 중", unit="건")

        # 배치별 처리
        for i in range(0, len(df), self.batch_size):
            batch_end = min(i + self.batch_size, len(df))
            batch_df = df.iloc[i:batch_end]

            try:
                summaries, costs = await self.process_batch(batch_df)

                all_summaries.extend(summaries)
                all_costs.extend(costs)

                # 진행바 업데이트
                pbar.update(len(batch_df))

                # 중간 저장 (5배치마다)
                batch_num = i // self.batch_size + 1
                if batch_num % 5 == 0:
                    temp_df = df.copy()
                    temp_df['요약'] = all_summaries + [''] * (len(df) - len(all_summaries))
                    temp_df['비용'] = all_costs + [0.0] * (len(df) - len(all_costs))
                    temp_df.to_csv(output_path, encoding='utf-8-sig', index=False)

            except Exception as e:
                batch_size = len(batch_df)
                all_summaries.extend([f"배치 오류: {str(e)[:30]}"] * batch_size)
                all_costs.extend([0.0] * batch_size)
                self.parsing_failures += batch_size
                pbar.update(len(batch_df))

        pbar.close()

        # 최종 결과 생성
        result_df = df.copy()
        result_df['요약'] = all_summaries
        result_df['비용'] = all_costs

        # 최종 저장
        result_df.to_csv(output_path, encoding='utf-8-sig', index=False)

        # 완료 메시지
        elapsed_time = time.time() - start_time
        self._print_completion_message(result_df, elapsed_time, output_path)

        return result_df

    def _print_completion_message(self, result_df, elapsed_time, output_path):
        """완료 메시지 출력"""
        total_count = len(result_df)
        success_count = self.parsing_successes
        fail_count = self.parsing_failures

        print(f"\n{'='*60}")
        print(f"✅ 뉴스 3줄 요약 완료!")
        print(f"{'='*60}")
        print(f"📊 총 {total_count:,}건 뉴스 요약 완료 (소요시간: {elapsed_time/60:.1f}분)")
        print(f"💰 총 비용: ${self.total_cost:.4f}")
        print(f"📁 결과 파일: {output_path}")
        print(f"\n📈 요약 성공률:")
        if total_count > 0:
            success_rate = (success_count / total_count) * 100
            print(f"   • 성공: {success_count}건 ({success_rate:.1f}%)")
            print(f"   • 실패: {fail_count}건 ({(fail_count/total_count)*100:.1f}%)")
        print(f"{'='*60}\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 4. 설정 및 실행
# ═══════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# 여기서 설정을 변경하세요
# ──────────────────────────────────────────────────────────────────────
MODEL = "gpt-4.1-mini"  # "gpt-4o-mini" 또는 "gpt-4o"
API_KEY = "Your-OpenAI-API-Key-Here"
INPUT_CSV_PATH = "Your-Input-CSV-Path-Here"
OUTPUT_CSV_PATH = "Your-Output-CSV-Path-Here"
# ──────────────────────────────────────────────────────────────────────

async def main():
    """메인 함수"""
    try:
        # CSV 파일 로드
        df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8-sig')

        # 필수 컬럼 확인
        if 'title' not in df.columns or 'text' not in df.columns:
            print("❌ title, text 컬럼 필요")
            return

        # 출력 경로 설정
        output_dir = os.path.dirname(OUTPUT_CSV_PATH) or '.'
        output_file = os.path.basename(OUTPUT_CSV_PATH)

        # 분석기 생성 및 실행
        analyzer = NewsSummaryAnalyzer(
            api_key=API_KEY,
            model=MODEL,
            output_dir=output_dir
        )

        await analyzer.analyze_dataframe(df, output_file)

    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없음: {INPUT_CSV_PATH}")
    except Exception as e:
        print(f"❌ 오류: {str(e)}")

def run_analysis():
    """분석 실행"""
    try:
        import nest_asyncio
        nest_asyncio.apply()
        asyncio.run(main())
    except ImportError:
        asyncio.run(main())

if __name__ == "__main__":
    run_analysis()